# Data & Model Parallelization 

## 1. Overview

Four basic strategies for scaling training across devices:

| Strategy | What is split | Split dimension |
|---|---|---|
| **Data Parallelism** | Training batch | across devices, model replicated |
| **Tensor Parallelism** | Model | "vertically" — feature dimension |
| **Pipeline Parallelism** | Model | "horizontally" — layer dimension |
| **FSDP (ZeRO)** | Model parameters + data | parameters sharded across devices |

- **Data parallelization**: simplest, most common; splits the *batch*, replicates the *model*.
- **Model parallelization**: splits the *model* itself.
  - **Pipeline parallelization**: split across **layers**.
  - **Tensor parallelization**: split across **feature dimensions**.

---

## 2. Data Parallelization

**Basic idea:**
1. Take a large batch, divide it into smaller sub-batches.
2. Distribute sub-batches across devices.
3. Each device processes its sub-batch in parallel (same model replica).
4. Aggregate results (e.g., gradients) across devices to update the model.

**Limitation:** every device stores a *full copy* of the model, gradients, and optimizer states → memory-inefficient for very large models.

### Memory-efficient variant: Fully-Sharded Data Parallelization (FSDP)
- Distributes the **model parameters** (not just data) across devices.
- Part of the **ZeRO optimizer** (Rajbhandari et al., 2020).

---

## 3. Memory Accounting Example — Mixed-Precision Training with Adam

Adam optimizer update rule (per parameter):

```
m_t = β1·m_{t-1} + (1-β1)·g_t        (1st moment)
v_t = β2·v_{t-1} + (1-β2)·g_t²       (2nd moment)
m̂_t = m_t / (1-β1^t)                 (bias-corrected)
v̂_t = v_t / (1-β2^t)
θ_t = θ_{t-1} - α · m̂_t / (√v̂_t + ε)
```

### Per-parameter memory cost (mixed precision)

| Component | Precision | Bytes |
|---|---|---|
| Parameter | bfloat16 | 2 |
| Gradient | bfloat16 | 2 |
| Optimizer 1st + 2nd moment | float32 | 8 |
| Parameter copy (for fp32 compute) | float32 | 4 |
| **Total** | | **16 bytes / parameter** |

This "K = 12" (optimizer state multiplier) + "2+2" (params+grads in bf16) pattern is used directly in the ZeRO memory formulas below.

---

## 4. The ZeRO-DP Model (Rajbhandari et al., 2020)

Notation:
- **Ψ** = model size (number of parameters) — example: Ψ = 7.5B
- **K** = memory multiplier for optimizer states — example: K = 12
- **N_d** = number of data-parallel devices — example: N_d = 64

| Stage | Memory formula | Example value |
|---|---|---|
| Baseline (standard DP) | (2 + 2 + K)·Ψ | 120 GB |
| P_os (optimizer state partitioning) | 2Ψ + 2Ψ + (K·Ψ)/N_d | 31.4 GB |
| P_os+g (+ gradient partitioning) | 2Ψ + (2+K)·Ψ/N_d | 16.6 GB |
| P_os+g+p (+ parameter partitioning = **FSDP**) | (2+2+K)·Ψ/N_d | 1.9 GB |

### Optimizer stages summary
- **P_os** — optimizer state partitioning: **4× memory reduction**, same communication volume as standard DP.
- **P_os+g** — optimizer state + gradient partitioning: **8× memory reduction**, same communication volume as standard DP.
- **P_os+g+p** — fully-sharded data parallelization: memory reduced by up to **1/N_d** (parameters divided across devices), but **~50% increase** in communication volume.

Reference implementation: [UvA DLC notebooks — FSDP](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/scaling/JAX/data_parallel_fsdp.html)

---

## 5. Why Move Beyond Data Parallelism?

- Data parallelism works well for **medium-sized models** with **large batch sizes**.
- As model size grows, the **per-device batch size shrinks** → inefficient accelerator usage:
  - Communication time becomes significant relative to compute time.
  - Small batches → poor utilization of compute resources (less parallelism per op).
- Solution: parallelize the **model** itself instead of (or in addition to) the data.
  - **Pipeline parallelization** — split across layers.
  - **Tensor parallelization** — split across feature dimensions.

---

## 6. Pipeline Parallelization

**Idea:** Parallelize the forward/backward pass across devices by splitting the model into **stages** (groups of consecutive layers), each stage placed on a different device.

- Output of stage *i* is passed to stage *i+1* (forward).
- Backward pass proceeds in reverse order (last stage → first stage).

### Example
Model with 16 layers, 4 devices → 4 stages of 4 layers each:
- Device 1: layers 1–4, Device 2: layers 5–8, etc.
- Forward: batch flows sequentially through devices 1→2→3→4.
- Backward: gradients flow 4→3→2→1.

### Pros / Cons

| Pro | Con |
|---|---|
| Each device holds only a subset of the model → lower memory, enables larger models | Increased inter-device communication (only adjacent stages communicate) |
| | Naive implementation → **pipeline bubble**: large idle time while devices wait for input/output |

### Mitigation 1: Micro-batching (Huang et al., 2019)
- Split each batch into smaller **micro-batches**, process sequentially through the pipeline.
- More devices are kept busy simultaneously → bubble shrinks.
- Trade-off: more frequent communication → more overhead.
- **Micro-batch size** is a trade-off between: pipeline bubble, communication overhead, and per-device utilization.

### Mitigation 2: Looping Pipelines (Lamy-Poirier, 2023)
- Standard pipelines can only start communicating gradients **after the last microbatch of a stage finishes** → idle time.
- **Looping / breadth-first schedule (PP_BF):**
  - Layers are interleaved across devices (e.g., device 1 gets layers 0,4,8,12; device 2 gets 1,5,9,13, …).
  - Each stage finishes all microbatches for its current layer before moving to the next.
  - In the backward pass, gradients for **later layers** finish first → their communication can **overlap** with computation of earlier layers' gradients.
  - Final communication of earliest layers is **cheaper** — only 1/n_loops of the gradients need to be sent at the end.
- Result: **small bubble, best overlap** vs. the non-looped GPipe-style schedule (**large bubble, poor overlap**).

---

## 7. Tensor Parallelization

**Idea:** Split the model **"horizontally" across feature dimensions** — each device processes a different subset of features; forward and backward passes are split across devices (all devices work on the *same* batch simultaneously).

### Advantages
- Can be applied **per-module/per-layer** → more flexible than pipeline parallelization.
- Can handle a single layer too large to fit on one device.
- **No pipeline bubble** — all devices work on the same data at the same time.

### Challenges
- Requires **overlapping computation with communication** and **minimizing communication volume**.
- Needs **more frequent communication** than pipeline or data parallelism.
- Requires **high-speed interconnects** (TPUs, GPUs with NVLink) → usually restricted to devices **within one node**.
- Practical pattern (e.g., Gemini v1): **tensor/model parallelism within a node**, **data parallelism across nodes**.

### Example: Matrix–Vector Multiplication

For `A·x = y`, with `A ∈ R^(n_y × n_x)`, `x ∈ R^(n_x)`, `y ∈ R^(n_y)`:
- Input `x` is sharded across devices: `x = (x0, x1, x2, x3)`.
- Goal: output also ends up sharded: `y = (y0, y1, y2, y3)`.

#### Two strategies to split matrix A

**Gather strategy** (splits **rows** of A):
- Input `x` is broadcast to all devices via `all_gather` (each device holds the full `x`).
- Each device independently computes: `y_i = Σ_j A_{i,j} x_j`.

**Scatter strategy** (splits **columns** of A):
- Each device computes a partial sum using only its slice of `x`: `y_j^i = A_{i,j} x_i`.
- Partial results are combined via `psum_scatter`: `y_i = Σ_k y_i^k`.

- Which strategy is more efficient depends on the sizes of `x`, `y`, and the hardware.
- **Advantage:** the strategy can be chosen **per layer**.

### Example: MLP Block in a Transformer

Typical MLP block: `Norm → Dense (up-project) → ActFn → Dense (down-project)`

- First part (up-projection to hidden dim `h0`, `dim(h0) > dim(x)`): use **gather** strategy.
- Second part (down-projection back to `dim(y) < dim(h0)`): use **scatter** strategy.
- Communication happens only at the **start and end** of the block (gather → ... → scatter), avoiding communication *within* the block → more efficient.

Flow: `x (sharded) → Gather → Norm → Dense → h0 (sharded) → ActFn → Dense → y (sharded) → Scatter → y (sharded)`

---

## 8. Quick Reference: Which Strategy When?

| Situation | Recommended approach |
|---|---|
| Model fits on one device, want faster training | Data parallelism |
| Model doesn't fit, memory is the bottleneck | FSDP / ZeRO (P_os+g+p) |
| Model too large even sharded, layer-level split acceptable | Pipeline parallelism (+ micro-batching / looping) |
| Single layer too large, or need fine-grained flexibility, have fast interconnect (NVLink/TPU) | Tensor parallelism |
| Training frontier-scale models | **Hybrid**: tensor/pipeline parallelism within a node + data parallelism across nodes |

---

## 9. Key References

- Rajbhandari et al., 2020 — *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models*
- Kingma & Ba, 2015 — *Adam: A Method for Stochastic Optimization*
- Huang et al., 2019 — *GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism*
- Lamy-Poirier, 2023 — Looping pipelines (breadth-first schedule)
- Gemini Team, 2023 — Gemini technical report (model + data parallel hybrid training)

### Further resources
- JAX implementations of all discussed models: [UvA DLC Notebooks — Scaling in JAX](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/scaling/JAX/overview.html)
- Book: [How to Scale Your Model](https://jax-ml.github.io/scaling-book/) — detailed account of TPUs, transformers, sharding, compute/communication cost, and a step-by-step LLaMA implementation.